# Training a layer detector

Trains one detector for one layer and writes it in the format the worker loads.
Set `LAYER` in the config cell and run top to bottom.

The worker reads five files from `build/worker/models/<layer>/`:

| File | Meaning |
|------|---------|
| `models.joblib` | the fitted estimator |
| `top_features.npy` | feature names, in the order the estimator was fitted on |
| `medians25.npy` | per-feature median, used to impute what a capture lacks |
| `scaler_mean25.npy` | per-feature mean |
| `scaler_std25.npy` | per-feature standard deviation |

Two rules this notebook enforces, each for a failure that has already happened
on this project:

1. **Only features the converter can produce.** A model trained on `Inbound`,
   `Unnamed: 0` or the ports scores well offline and is useless live, because
   CICFlowMeter never emits those columns — the worker would impute a median
   for every one of them.
2. **The exported model must fire on testbed traffic before it ships.** A model
   with 0.9999 AUC on CIC-DDoS2019 scored a real 59,488-flow SYN flood at
   p = 0.0005 on this testbed. Section 8 is the gate that catches that, and it
   needs captures from the testbed itself — `training/collect_testbed_flows.py`
   produces them.

## 0. On Colab

Run this cell first there; skip it when running inside a clone of the repo. It
brings in the worker package, because every cell below builds its feature matrix
with the serving code rather than a copy of it.

In [ ]:
import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = Path("/content/DOS-Detection-in-SDN-Docker-Network")
    if not REPO.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/teodoramz/DOS-Detection-in-SDN-Docker-Network.git",
             str(REPO)],
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "xgboost", "imbalanced-learn"], check=True)

    from google.colab import drive
    drive.mount("/content/drive")
    # Put the CIC-DDoS2019 CSVs here, or change the path.
    DATA_DIR = Path("/content/drive/MyDrive/ddos/datasets")
    print("repo:", REPO, "| datasets:", DATA_DIR, "|", DATA_DIR.is_dir())
else:
    print("not on Colab; the next cell locates the repo itself")

## 1. Configuration

In [ ]:
from pathlib import Path

LAYER = "inter"              # "top", "inter" or "bottom"
if "REPO" not in dir():
    REPO = Path.cwd().parent if Path.cwd().name == "training" else Path.cwd()

if "DATA_DIR" not in dir():
    DATA_DIR = Path("~/datasets").expanduser()  # the CIC-DDoS2019 CSVs
OUT_DIR = REPO / "build" / "worker" / "models" / LAYER
REPORT_DIR = REPO / "training" / "reports" / LAYER

# Flows captured on the testbed, labelled 0/1. Produced by
# training/collect_testbed_flows.py. Section 8 is skipped without them, and
# then nothing has checked the model against the traffic it will actually see.
TESTBED_CSV = REPO / "training" / "data" / f"testbed_{LAYER}.csv.gz"

ATTACK_LABELS = {
    "top": ["DrDoS_DNS"],
    "inter": ["Syn", "DrDoS_UDP", "UDP-lag", "DrDoS_SSDP", "DrDoS_NTP"],
    "bottom": ["WebDDoS", "DrDoS_LDAP", "DrDoS_MSSQL", "DrDoS_NetBIOS", "DrDoS_SNMP", "TFTP"],
}[LAYER]

TOP_N = 25                   # the deployed vectors are named *25 but any count works
MAX_PER_CLASS = 300_000
TEST_SIZE = 0.25
RANDOM_STATE = 42

# The deployed alerting rule, mirrored here so offline numbers mean something.
ALERT_THRESHOLD = 0.5
ALERT_MIN_FLOWS = 10

# Split by source address instead of at random. Keeps a host's flows on one
# side of the split, so a model cannot score well by memorising addresses.
GROUP_BY_SOURCE = True

OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print("layer", LAYER, "->", OUT_DIR)

## 2. Imports, and the serving code as the single source of truth

In [ ]:
import sys, json, time, warnings
import numpy as np
import pandas as pd

sys.path.insert(0, str(REPO / "build" / "worker"))
from ddos_worker.features import (          # noqa: E402
    COLUMN_MAPPING, DUPLICATE_HEADER_COLUMN, FeatureSpec,
    build_matrix, ensure_duplicate_header_column, rename_columns,
)

from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import (
    ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    average_precision_score, classification_report, confusion_matrix,
    f1_score, roc_auc_score,
)
import joblib

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None
    print("xgboost not installed; it will be skipped")

warnings.filterwarnings("ignore", category=UserWarning)
print("pandas", pd.__version__, "| numpy", np.__version__)

## 3. What the converter can actually produce

`COLUMN_MAPPING` is the worker's own translation from CICFlowMeter names to the
dataset names. Its values are therefore the complete list of columns that exist
at serving time. Anything outside it is unavailable live, and the identifiers
below are dropped as well: addresses and timestamps identify the testbed rather
than the attack, and ports let a model memorise a scenario.

In [ ]:
NOT_FEATURES = {
    "Flow ID", "Source IP", "Destination IP", "Timestamp", "Label",
    "Source Port", "Destination Port", "Protocol",
    "SimillarHTTP", "Unnamed: 0", "Inbound",
}

SERVABLE = sorted((set(COLUMN_MAPPING.values()) | {DUPLICATE_HEADER_COLUMN}) - NOT_FEATURES)
print(f"{len(SERVABLE)} candidate features the converter produces")
print(f"dropped as identifiers or unavailable: {sorted(NOT_FEATURES)}")

## 4. Load the dataset

In [ ]:
files = sorted(DATA_DIR.glob("*.csv"))
assert files, f"no CSVs under {DATA_DIR}"

def load(path, wanted):
    header = pd.read_csv(path, nrows=0)
    header.columns = [c.strip() for c in header.columns]
    label_col = "Label" if "Label" in header.columns else header.columns[-1]
    usecols = [c for c in wanted if c in header.columns] + [label_col]
    parts = []
    for chunk in pd.read_csv(path, chunksize=200_000, low_memory=False,
                             skipinitialspace=True, usecols=lambda c: c.strip() in set(usecols)):
        chunk.columns = [c.strip() for c in chunk.columns]
        chunk = chunk.rename(columns={label_col: "Label"})
        parts.append(chunk[chunk.Label.isin(ATTACK_LABELS + ["BENIGN"])])
    return pd.concat(parts, ignore_index=True) if parts else None

wanted = set(SERVABLE) | {"Source IP"}
frames = []
for path in files:
    t0 = time.perf_counter()
    part = load(path, wanted)
    if part is None or part.empty:
        continue
    frames.append(part)
    print(f"{path.name}: {len(part):>9,} rows  {time.perf_counter()-t0:5.1f}s")

df = pd.concat(frames, ignore_index=True)
df["y"] = (df.Label != "BENIGN").astype(int)
df = df.replace([np.inf, -np.inf], np.nan)
print("\ntotal", f"{len(df):,}", "| attack share", round(df.y.mean(), 4))
print(df.Label.value_counts())

In [ ]:
balanced = [g.sample(min(len(g), MAX_PER_CLASS), random_state=RANDOM_STATE)
            for _, g in df.groupby("y")]
df = pd.concat(balanced, ignore_index=True).sample(frac=1, random_state=RANDOM_STATE)
print("after subsampling:", f"{len(df):,}", "| attack share", round(df.y.mean(), 4))

## 5. Split

With `GROUP_BY_SOURCE`, every flow from one source address lands wholly in
train or wholly in test. A random row split lets the same host appear on both
sides, which inflates every score below.

In [ ]:
feature_cols = [c for c in SERVABLE if c in df.columns]
X_all = df[feature_cols]
y_all = df.y.to_numpy()

if GROUP_BY_SOURCE and "Source IP" in df.columns:
    groups = df["Source IP"].astype(str).to_numpy()
    splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_idx, test_idx = next(splitter.split(X_all, y_all, groups))
    X_tr, X_te = X_all.iloc[train_idx], X_all.iloc[test_idx]
    y_tr, y_te = y_all[train_idx], y_all[test_idx]
    src_te = groups[test_idx]
    print("grouped split by source address")
else:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_all, y_all, test_size=TEST_SIZE, stratify=y_all, random_state=RANDOM_STATE)
    src_te = np.array(["unknown"] * len(X_te))
    print("stratified random split")

print(f"train {X_tr.shape} attack {y_tr.mean():.3f} | test {X_te.shape} attack {y_te.mean():.3f}")

## 6. Feature selection and the preprocessing vectors

Medians, mean and standard deviation come from the training split only, and are
written in the same order as the selected names. The worker reads that order
from `top_features.npy` and applies these three vectors in exactly this way, so
a mismatch here is a silent scrambling of every input at serving time.

In [ ]:
medians_all = X_tr.median(numeric_only=True)
Xtr_imputed = X_tr.fillna(medians_all)[feature_cols]

f_scores = pd.Series(
    SelectKBest(f_classif, k="all").fit(Xtr_imputed, y_tr).scores_,
    index=feature_cols,
).fillna(0).rank(ascending=False)

et = ExtraTreesClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
et.fit(Xtr_imputed, y_tr)
et_rank = pd.Series(et.feature_importances_, index=feature_cols).rank(ascending=False)

ranking = (f_scores + et_rank).sort_values()
FEATURES = ranking.head(TOP_N).index.tolist()
print(f"selected {len(FEATURES)}:")
for i, name in enumerate(FEATURES, 1):
    print(f"  {i:2d}. {name}")

In [ ]:
medians = X_tr[FEATURES].median().to_numpy(dtype=float)
mean = X_tr[FEATURES].fillna(X_tr[FEATURES].median()).mean().to_numpy(dtype=float)
std = X_tr[FEATURES].fillna(X_tr[FEATURES].median()).std(ddof=0).to_numpy(dtype=float)

spec = FeatureSpec(order=FEATURES, medians=medians, mean=mean, std=std)

def to_matrix(frame):
    """Exactly the transformation the worker applies, via the worker's own code."""
    return build_matrix(frame, spec)[0]

Xtr_m, Xte_m = to_matrix(X_tr), to_matrix(X_te)
print("train matrix", Xtr_m.shape, "| test matrix", Xte_m.shape)
print("any non-finite:", bool(~np.isfinite(Xtr_m).all()), bool(~np.isfinite(Xte_m).all()))

## 7. Candidates

Each is fitted on the same matrix, so the comparison is about the algorithm.
`alert_f1` applies the deployed rule — a source alerts when at least
`ALERT_MIN_FLOWS` of its flows score at or above `ALERT_THRESHOLD` — which is
what the controller acts on. A model can have an excellent per-flow F1 and
still never raise an alert.

In [ ]:
candidates = {
    "RandomForest": RandomForestClassifier(
        n_estimators=300, n_jobs=-1, class_weight="balanced_subsample",
        random_state=RANDOM_STATE),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=300, n_jobs=-1, class_weight="balanced",
        random_state=RANDOM_STATE),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=300, random_state=RANDOM_STATE),
    "LogisticRegression": LogisticRegression(
        max_iter=2000, class_weight="balanced", n_jobs=-1),
    "MLP": MLPClassifier(hidden_layer_sizes=(120, 60), max_iter=60,
                         random_state=RANDOM_STATE),
}
if XGBClassifier is not None:
    pos = max(float((y_tr == 0).sum()) / max(float((y_tr == 1).sum()), 1.0), 1.0)
    candidates["XGBoost"] = XGBClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.1, subsample=0.8,
        colsample_bytree=0.8, tree_method="hist", n_jobs=-1,
        eval_metric="aucpr", scale_pos_weight=pos, random_state=RANDOM_STATE)


def probabilities(est, X):
    if hasattr(est, "predict_proba"):
        p = np.asarray(est.predict_proba(X), dtype=float)
        return p[:, 1] if p.ndim == 2 and p.shape[1] > 1 else p.ravel()
    if hasattr(est, "decision_function"):
        m = np.asarray(est.decision_function(X), dtype=float).ravel()
        return 1.0 / (1.0 + np.exp(-m))
    return np.asarray(est.predict(X), dtype=float).ravel()


def alert_scores(sources, probs, truth):
    """Apply the deployed rule per source and score the alerts, not the flows."""
    frame = pd.DataFrame({"src": sources, "p": probs, "y": truth})
    per_source = frame.groupby("src").agg(
        malicious=("p", lambda s: int((s >= ALERT_THRESHOLD).sum())),
        attack=("y", "max"),
    )
    predicted = (per_source.malicious >= ALERT_MIN_FLOWS).astype(int)
    if per_source.attack.nunique() < 2:
        return float("nan"), int(predicted.sum())
    return f1_score(per_source.attack, predicted), int(predicted.sum())


rows, fitted = [], {}
for name, est in candidates.items():
    t0 = time.perf_counter()
    est.fit(Xtr_m, y_tr)
    p = probabilities(est, Xte_m)
    hard = (p >= ALERT_THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, hard, labels=[0, 1]).ravel()
    a_f1, a_n = alert_scores(src_te, p, y_te)
    rows.append(dict(
        model=name, auc=roc_auc_score(y_te, p), pr_auc=average_precision_score(y_te, p),
        f1=f1_score(y_te, hard), fpr=fp / max(fp + tn, 1),
        recall=tp / max(tp + fn, 1), alert_f1=a_f1, alerting_sources=a_n,
        fit_s=round(time.perf_counter() - t0, 1),
    ))
    fitted[name] = est
    print(f"{name:22s} auc={rows[-1]['auc']:.5f} pr_auc={rows[-1]['pr_auc']:.5f} "
          f"f1={rows[-1]['f1']:.5f} fpr={rows[-1]['fpr']:.2e} alert_f1={a_f1:.4f}")

results = pd.DataFrame(rows).sort_values(["alert_f1", "pr_auc"], ascending=False)
results.to_csv(REPORT_DIR / "candidates.csv", index=False)
results

## 8. The gate: does it fire on testbed traffic?

CIC-DDoS2019 was captured on different hardware with a different CICFlowMeter
build, so a column can carry the same name and a different distribution. This
cell scores flows taken from this testbed. A model that cannot reach the
threshold here will never block anything, whatever its offline AUC says.

Produce the file with, on host2:

```bash
docker cp training/collect_testbed_flows.py worker2:/tmp/

# while the attack is running
docker exec worker2 python3 /tmp/collect_testbed_flows.py --label 1 --last 3 --out /tmp/attack.csv
# while only legitimate traffic is flowing
docker exec worker2 python3 /tmp/collect_testbed_flows.py --label 0 --last 3 --out /tmp/benign.csv
```

Launch the attack **without** `SPOOF_SOURCE` for this: the source is then the
host's own whitelisted address, so no drop rule lands mid-collection and cuts
off the very traffic being captured. `training/data/testbed_<layer>.csv.gz`
already holds one such set per layer, collected this way.
```

then concatenate the two into `training/data/testbed_<layer>.csv`.

In [ ]:
testbed = None
if not TESTBED_CSV.is_file() and TESTBED_CSV.with_suffix("").is_file():
    TESTBED_CSV = TESTBED_CSV.with_suffix("")      # an uncompressed set is fine too
if TESTBED_CSV.is_file():
    testbed = pd.read_csv(TESTBED_CSV, low_memory=False)
    testbed = ensure_duplicate_header_column(rename_columns(testbed))
    testbed = testbed.replace([np.inf, -np.inf], np.nan)
    print(f"{len(testbed):,} testbed flows, attack share {testbed.y.mean():.3f}")
else:
    print(f"!! {TESTBED_CSV} absent — the gate cannot run and nothing has")
    print("!! checked these models against the traffic they will serve")

gate = []
if testbed is not None:
    Xt = to_matrix(testbed)
    yt = testbed.y.to_numpy()
    srct = testbed.get("Source IP", pd.Series(["unknown"] * len(testbed))).astype(str).to_numpy()
    for name, est in fitted.items():
        p = probabilities(est, Xt)
        a_f1, a_n = alert_scores(srct, p, yt)
        attack_p = p[yt == 1]
        gate.append(dict(
            model=name,
            attack_p50=float(np.percentile(attack_p, 50)) if attack_p.size else float("nan"),
            attack_pmax=float(attack_p.max()) if attack_p.size else float("nan"),
            attack_over_threshold=int((attack_p >= ALERT_THRESHOLD).sum()),
            benign_over_threshold=int((p[yt == 0] >= ALERT_THRESHOLD).sum()),
            alert_f1=a_f1, alerting_sources=a_n,
            would_fire=bool(a_n > 0),
        ))
    gate = pd.DataFrame(gate).sort_values(["alert_f1", "attack_pmax"], ascending=False)
    gate.to_csv(REPORT_DIR / "testbed_gate.csv", index=False)
    if not gate.would_fire.any():
        print("\nNONE of these models would raise a single alert on testbed traffic.")
        print("Exporting one now ships a detector that cannot block. Fix the")
        print("features or train on testbed flows before continuing.")
    display(gate)

## 9. Pick one and export it

In [ ]:
if len(gate) if isinstance(gate, pd.DataFrame) else 0:
    firing = gate[gate.would_fire]
    BEST = (firing.iloc[0].model if len(firing) else gate.iloc[0].model)
else:
    BEST = results.iloc[0].model
print("exporting", BEST)

estimator = fitted[BEST]
np.save(OUT_DIR / "top_features.npy", np.array(FEATURES, dtype=object), allow_pickle=True)
np.save(OUT_DIR / "medians25.npy", medians)
np.save(OUT_DIR / "scaler_mean25.npy", mean)
np.save(OUT_DIR / "scaler_std25.npy", std)
joblib.dump(estimator, OUT_DIR / "models.joblib")

report = {
    "layer": LAYER,
    "model": BEST,
    "algorithm": type(estimator).__name__,
    "features": FEATURES,
    "attack_labels": ATTACK_LABELS,
    "grouped_split": bool(GROUP_BY_SOURCE),
    "holdout": results[results.model == BEST].to_dict("records")[0],
    "testbed_gate": (gate[gate.model == BEST].to_dict("records")[0]
                     if isinstance(gate, pd.DataFrame) and len(gate) else None),
    "trained_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
(REPORT_DIR / "training_report.json").write_text(json.dumps(report, indent=2, default=str))
print(json.dumps({k: v for k, v in report.items() if k != "features"}, indent=2, default=str))

Only a bare estimator is written. A pipeline carrying a class defined in this
notebook cannot be unpickled anywhere else — `pipe_SMOTE_XGB.joblib` from the
earlier runs fails with `Can't get attribute 'TopFeatureSelector'`, which is why
it was never deployable. Keep custom transformers out of the exported object.

## 10. Round trip

Reload through the worker's own code and confirm the scores survive, and that
every feature resolves from real converter output.

In [ ]:
from ddos_worker.detectors import build_detector     # noqa: E402

detector = build_detector(LAYER, OUT_DIR, "model")
print("detector:", detector.name, "| preprocessing:", detector.preprocessing)

reloaded = FeatureSpec.load(OUT_DIR)
assert reloaded.order == FEATURES, "the exported feature order does not round trip"
assert np.allclose(reloaded.medians, medians)
assert np.allclose(reloaded.mean, mean)
assert np.allclose(reloaded.std, std)

sample = (testbed if testbed is not None else X_te.head(2000))
through_detector = detector.score(sample)
print("scored", len(through_detector), "rows | max p =", round(float(through_detector.max()), 5))

header = (REPO / "tests" / "fixtures" / "cicflowmeter4_header.csv").read_text().strip()
names = [c.strip() for c in header.split(",")]
available = set(ensure_duplicate_header_column(
    rename_columns(pd.DataFrame({n: [0] for n in names}))).columns)
missing = [f for f in FEATURES if f not in available]
assert not missing, f"the converter cannot produce {missing}"
print("all", len(FEATURES), "features resolve from real CICFlowMeter output")

## 11. Deploy

```bash
git add build/worker/models/<layer> && git commit -m "Retrain the <layer> detector"
# on host2
sudo DDOS_DETECTION_HOME=$PWD ./deploy/host2.sh
docker logs worker2 | grep detector=
```

The worker names the algorithm it loaded, and that name reaches every alert, so
`inter-randomforestclassifier` in the controller log is the confirmation that
this model is the one scoring traffic. A layer whose artifacts are missing falls
back to the rule-based placeholder and says so; `DETECTOR=model` makes that
fatal instead.

## 12. Taking the artifacts off Colab

The five files go into `build/worker/models/<layer>/` in the repo, then host2 is
redeployed. Keep the whole set together: the estimator and its three vectors are
only valid for the feature order they were written with.

In [ ]:
if IN_COLAB:
    import shutil
    bundle = Path(f"/content/{LAYER}_model")
    shutil.rmtree(bundle, ignore_errors=True)
    shutil.copytree(OUT_DIR, bundle)
    archive = shutil.make_archive(str(bundle), "zip", bundle)
    from google.colab import files
    files.download(archive)
    print("unzip into build/worker/models/" + LAYER + " and redeploy host2")